# 【学習用・解説付き写し】s6e8 smartphone addiction eda fast

- **コンペ**: [Predicting Smartphone Addiction — Playground Series S6E8](https://www.kaggle.com/competitions/playground-series-s6e8)（Playground・3,372チーム・**残り1日**）
- **原著notebook**: [s6e8 smartphone addiction eda fast](https://www.kaggle.com/code/lamhuy8904/s6e8-smartphone-addiction-eda-fast)
- **原著者**: LÂM HUY (lamhuy8904)
- **スコア**: Public LB **0.97125**（V4）・15 votes・Bronze
- **ライセンス**: Apache 2.0

> ⚠️ これは**学習目的の解説付き写し**です。原著コードは変更せず、各コードセルの直前に日本語解説Markdownセルを挿入しています。**未実行**のため出力は含まれていません。

---

## 手法の概要（1段落）

スマホ依存の有無（2値）をROC AUCで予測するタスク。合成データ（CTGAN生成）特有の性質を突く特徴量エンジニアリング（**小数第1位などの「桁アーティファクト」抽出**、時間予算の物理的整合性から作る比率特徴、train+testを合わせた**transductiveな頻度エンコーディング**）を施し、LightGBM / XGBoost / CatBoost の3モデルを Stratified 5-Fold で学習。OOF予測を**ロジット→パーセンタイルランク**に変換し、SLSQPでAUC最大化する重みを求めてブレンドする。最後に、入力にアタッチされた**他人の公開submission CSVを「アンカー」として自動検出**し、`0.725 × 1本目 + 0.225 × 2本目 + 0.05 × 自前ブレンド` という比率で融合、さらに2件のIDを0/1に決め打ちする "Duplicate Magic" を適用して提出する。

> 📌 **本notebookから学ぶべき最大のポイントは、後半の「公開アンカー融合」が自前モデルの寄与をわずか5%にしている**という事実です。LB 0.97125 という数字の大部分は他人の予測に由来します。手法自体は勉強になりますが、**スコアの帰属を正しく読む練習**として扱ってください（詳細は該当セルの解説とREADMEの改善点考察を参照）。

---

## 評価指標

### タスク
表形式データから `addicted_label`（スマホ依存 = 1 / 非依存 = 0）を予測する2値分類。

### 指標: ROC AUC（Area Under the ROC Curve）
- 予測スコアで全サンプルを並べたとき、**ランダムに選んだ正例が、ランダムに選んだ負例より高いスコアを持つ確率**に等しい。0.5がランダム、1.0が完璧。
- **順位（ランキング）だけを見る指標**であり、予測値の絶対的な大きさ（キャリブレーション）は一切問われない。0.3と0.7の予測を両方2倍しても、順序が変わらなければAUCは不変。

### なぜこの指標か
- Playground Seriesの合成データは**クラス不均衡**を含むことが多く、Accuracyだと「全部多数派と答える」だけで高得点になってしまう。AUCは不均衡に対して頑健。
- 「依存かどうか」を白黒つけるより「依存リスクが高い順に並べる」ほうが実務的に有用、という設定にも合致する。

### この手法が指標をどう最適化しているか
- **すべての予測をパーセンタイルランクに変換してからブレンド**している（`percentile_rank(to_logit(p))`）。AUCは順位しか見ないので、スケールの違う3モデルを生確率のまま平均すると、分散の大きいモデルに引きずられる。ランク化すれば各モデルが対等に効く。**AUCコンペでのランク平均は定石**。
- ブレンド重みの最適化目的関数が `-roc_auc_score(y, blend)`、つまり**評価指標そのものを直接最大化**している（`scipy.optimize.minimize` の SLSQP）。代理損失（logloss等）ではなく本番指標を最適化する典型例。
- 各モデルの `early_stopping` も `eval_metric='auc'` 基準（訓練セル参照）。
- ただし後述の通り、**最終スコアの95%は外部アンカーから来ている**ため、この最適化が実際のLBに与えた寄与は限定的です。

---


# Playground Series S6E8: Smartphone Addiction Prediction
### Master Grandmaster Edition: Triple-Model Zoo, OOF Bayesian Target Encodings, Decimal Lattice, Model Checkpoint Saving & Calibrated Rank Fusion (LB 0.97125+)

---

### Key Workflow Highlights:
1. **Bayesian Out-of-Fold Target Encoding**: Applied across continuous and discrete attributes with Bayesian smoothing.
2. **Decimal Lattice Geometry**: Extracting fractional components (`frac`) and first decimal digits (`d1`).
3. **Transductive Global Frequencies**: Full 987k population density mapping.
4. **Triple-Model Zoo Training & Serialization**:
   - LightGBM GBDT (Tuned 3500 trees) -> Serialized to `.joblib` / `.txt`
   - XGBoost Hist GPU (3500 trees) -> Serialized to `.json`
   - CatBoost GPU (3500 iterations) -> Serialized to `.cbm`
5. **Calibrated Regime-Rank Fusion**: Optimal percentile rank ensembling achieving **LB 0.97125+**.
6. **Deterministic Duplicate Magic Overrides**: Exact label corrections for duplicate observations.

---

## 【解説】セル0: ライブラリのインポートと環境設定

**何をしているか（What）**
数値計算（numpy / pandas）、可視化（matplotlib / seaborn）、勾配ブースティング3種（LightGBM / XGBoost / CatBoost）、統計・最適化（`scipy.stats.rankdata`, `scipy.optimize.minimize`）、モデル保存（joblib）をまとめて読み込み、乱数シードを42に固定してGPUの有無を検出します。

**なぜそうするのか（Why）**
- **シード固定**は再現性の基本。同じコードで同じ結果が出ないと、改善が本物かノイズか判別できません。
- **勾配ブースティング3種を並べて用意**するのは表形式コンペの定番。LightGBM（葉ごと成長で高速）、XGBoost（レベルごと成長で安定）、CatBoost（カテゴリ変数の順序付きTarget Encodingが内蔵）は誤りの傾向が異なるため、アンサンブルで相補的に効きます。
- `rankdata` と `minimize` は後半のランク融合と重み最適化で使います。ここで入っていることが「アンサンブル前提の設計」を予告しています。

**用語メモ**
- **勾配ブースティング (GBDT)**: 浅い決定木を「前の木の誤差を修正する」形で何百本も足し合わせる手法。表形式データでは今なおニューラルネットより強いことが多い。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import glob
import time
import gc
import joblib
from scipy.stats import rankdata, spearmanr
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 150)

SEED = 42
np.random.seed(SEED)

# Detect GPU availability
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU Available:', HAS_GPU)
if HAS_GPU:
    print('Device:', torch.cuda.get_device_name(0))

os.makedirs('saved_models', exist_ok=True)

## 1. Dynamic Data Ingestion

## 【解説】セル1: 動的なデータ読み込み

**何をしているか（What）**
`glob` で `/kaggle/input/` 以下を再帰探索し、`train.csv` / `test.csv` / `sample_submission.csv` を**パスを決め打ちせずに**見つけて読み込みます。見つからなければローカルの `./data/` も探し、それでも無ければ例外。

**なぜそうするのか（Why）**
Kaggleの入力パスはコンペのslugやアタッチしたデータセット名によって変わります。ハードコードすると、notebookをフォークした人が最初のセルでエラーになる。`glob` で探す方式は**フォークされることを前提にした親切な設計**で、公開notebookでよく見られます。

**初心者向け補足**: `glob.glob(pattern, recursive=True)` の `**` は「任意の深さのサブディレクトリ」を意味します。


In [ ]:
def find_file(pattern):
    matches = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not matches:
        matches = glob.glob(f'./data/**/{pattern}', recursive=True)
    if not matches:
        matches = glob.glob(f'./**/{pattern}', recursive=True)
    if not matches:
        raise FileNotFoundError(f'File matching {pattern} not found.')
    return sorted(matches)[0]

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sub_path = find_file('sample_submission.csv')

print('Train path:', train_path)
print('Test path:', test_path)
print('Sample submission path:', sub_path)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

TARGET = 'addicted_label'
y = train[TARGET].values

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Target mean: {y.mean():.4f}')

## 2. Advanced Feature Engineering Pipeline

## 【解説】セル2: 特徴量エンジニアリング（本notebookの中核その1）

**何をしているか（What）**
生の12列から40以上の派生特徴を作ります。カテゴリは大きく5つ：

1. **欠損カウント** `missing_count`: 各行で何個の値が欠けているか。
2. **小数格子（Decimal Lattice）**: `frac_x = x - floor(x)`（小数部）と `d1_x = floor(x*10) % 10`（小数第1位の数字）。コメントに *"Artifacts of Synthetic CTGAN Generator"* とある通り、**合成データ生成器が残した痕跡**を拾う特徴。
3. **時間予算の物理量**: `active_waking_hours = 24 - sleep_hours`、`screen_to_waking_ratio`、`unaccounted_screen_time`（スクリーン時間 − SNS − ゲーム − 仕事/勉強）など。
4. **利用強度**: `notification_response_rate = app_opens / notifications`、`opens_per_screen_hour` など、**割り算で作る密度・効率の指標**。
5. **transductive頻度**: `train + test を連結した全体` での値の出現頻度 `freq_x`。

**なぜそうするのか（Why）**
- **小数格子が効く理由**: Playground Seriesのデータは実データをCTGANなどで合成したものです。生成器は連続値を独特の量子化パターンで出すため、**小数部の分布がクラスによって微妙に偏る**ことがあります。これは「現実の因果」ではなく「データ生成過程のリーク」に近い特徴です。スコアは上がりますが、**実務の予測モデルには一切転用できない**ことを理解して使うべきです。
- **比率特徴が効く理由**: GBDTは決定木の集まりなので、`a / b` のような**特徴間の除算を表現するのが非常に苦手**です（軸に平行な分割しかできないため、比率を近似するには階段状に大量の分割が必要）。人間が先に割り算しておくと、木は1回の分割で使えます。これは表形式FEの最重要原則の1つ。
- **transductive頻度エンコーディングの理由**: 「その値が全体で何回出現するか」は、合成データでは**元データの再サンプリング構造を反映**します。testを含めて数えるのがポイント（transductive = テスト特徴量を使ってよい設定）。ただし、これはtestの分布を覗いているので、**本番運用では使えない**手法です。

**用語メモ**
- **transductive learning**: 予測対象のテスト入力（ラベルは無し）を学習時に利用してよい設定。Kaggleでは合法だが、実運用では未来のデータが手に入らないので不可。
- **eps = 1e-4**: ゼロ除算を避けるための微小量。分母に必ず足す。


In [ ]:
def build_base_features(df_train, df_test):
    df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    eps = 1e-4
    
    # Missing Value Counter
    raw_cols = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                'work_study_hours', 'sleep_hours', 'notifications_per_day',
                'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact']
    df_all['missing_count'] = df_all[raw_cols].isnull().sum(axis=1)
    
    # Decimal Lattice (Artifacts of Synthetic CTGAN Generator)
    FRAC_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                 'work_study_hours', 'sleep_hours', 'weekend_screen_time']
    for c in FRAC_COLS:
        v = df_all[c].values
        df_all[f'frac_{c}'] = (v - np.floor(v)).astype(np.float32)
        df_all[f'd1_{c}'] = (np.floor(v * 10) % 10).astype(np.float32)
        
    # Time Budget Physics
    df_all['active_waking_hours'] = (24.0 - df_all['sleep_hours'].fillna(7.0)).astype(np.float32)
    df_all['screen_to_waking_ratio'] = (df_all['daily_screen_time_hours'] / (df_all['active_waking_hours'] + eps)).astype(np.float32)
    df_all['non_screen_waking_hours'] = (df_all['active_waking_hours'] - df_all['daily_screen_time_hours'].fillna(0)).astype(np.float32)
    
    df_all['total_leisure_hours'] = (df_all['social_media_hours'].fillna(0) + df_all['gaming_hours'].fillna(0)).astype(np.float32)
    df_all['unaccounted_screen_time'] = (df_all['daily_screen_time_hours'] - (df_all['social_media_hours'].fillna(0) + df_all['gaming_hours'].fillna(0) + df_all['work_study_hours'].fillna(0))).astype(np.float32)
    df_all['social_gaming_to_work_ratio'] = (df_all['total_leisure_hours'] / (df_all['work_study_hours'] + eps)).astype(np.float32)
    
    df_all['weekend_weekday_diff'] = (df_all['weekend_screen_time'] - df_all['daily_screen_time_hours']).astype(np.float32)
    df_all['weekend_weekday_ratio'] = (df_all['weekend_screen_time'] / (df_all['daily_screen_time_hours'] + eps)).astype(np.float32)
    df_all['screen_to_sleep_ratio'] = (df_all['daily_screen_time_hours'] / (df_all['sleep_hours'] + eps)).astype(np.float32)
    
    # Usage Intensity
    df_all['notification_response_rate'] = (df_all['app_opens_per_day'] / (df_all['notifications_per_day'] + eps)).astype(np.float32)
    df_all['opens_per_screen_hour'] = (df_all['app_opens_per_day'] / (df_all['daily_screen_time_hours'] + eps)).astype(np.float32)
    df_all['notifications_per_screen_hour'] = (df_all['notifications_per_day'] / (df_all['daily_screen_time_hours'] + eps)).astype(np.float32)
    df_all['opens_per_screen_minute'] = (df_all['app_opens_per_day'] / (df_all['daily_screen_time_hours'] * 60.0 + eps)).astype(np.float32)
    
    # Transductive Frequency Counts
    COUNT_COLS = ['age', 'notifications_per_day', 'app_opens_per_day', 'gender', 'stress_level', 'academic_work_impact']
    for c in COUNT_COLS:
        freq = df_all[c].astype(str).map(df_all[c].astype(str).value_counts(normalize=True))
        df_all[f'freq_{c}'] = freq.astype(np.float32)
        
    # Ordinal Mappings
    stress_map = {'Low': 0, 'Medium': 1, 'High': 2}
    df_all['stress_level_num'] = df_all['stress_level'].map(stress_map).fillna(-1).astype(np.float32)
    impact_map = {'No': 0, 'Yes': 1}
    df_all['academic_impact_num'] = df_all['academic_work_impact'].map(impact_map).fillna(-1).astype(np.float32)
    gender_map = {'Male': 0, 'Female': 1, 'Other': 2}
    df_all['gender_num'] = df_all['gender'].map(gender_map).fillna(-1).astype(np.float32)
    
    # Clean Categorical Types
    cat_columns = ['gender', 'stress_level', 'academic_work_impact']
    for c in cat_columns:
        df_all[c] = df_all[c].astype(object).fillna('__missing__').astype('category')
        
    train_base = df_all.iloc[:len(df_train)].copy()
    test_base = df_all.iloc[len(df_train):].copy().drop(columns=[TARGET])
    
    return train_base, test_base

train_base, test_base = build_base_features(train, test)
print(f'Base features created: {train_base.shape[1]} columns')

## 3. Out-of-Fold Target Encoding Engine

## 【解説】セル3: Out-of-Fold ベイジアン Target Encoding

**何をしているか（What）**
`age`, `gender`, `stress_level`, `academic_work_impact`, `notifications_per_day`, `app_opens_per_day` の6列について、**カテゴリ値ごとの目的変数の平均**を特徴量に変換します。ただし素朴な平均ではなく、

```
encoded = (カテゴリ内の合計 + 全体平均 × SMOOTHING) / (カテゴリの件数 + SMOOTHING)
```

というベイジアン平滑化（`SMOOTHING = 20.0`）を掛け、さらに**foldごとに学習用インデックスだけから計算**（Out-of-Fold）しています。

**なぜそうするのか（Why）**
- **Target Encodingとは**: カテゴリ変数を「そのカテゴリでの目的変数の平均」に置き換える手法。one-hotと違って次元が増えず、高カーディナリティ（値の種類が多い）列に強い。
- **なぜOOFが必須か**: 自分自身のラベルを使って自分の特徴量を作ると、**目的変数リーク**が起きます。訓練データ上では完璧に効き、CVもLBも一致しない典型的な失敗。foldを切って「自分が入っていないfoldから計算した値」を使うことで、これを防ぎます。
- **なぜ平滑化するか**: 件数が3件しかないカテゴリの平均は極端な値（0.0や1.0）になりがちで、過学習の温床です。全体平均に向かって引き戻す（**縮小推定 / shrinkage**）ことで安定させます。`SMOOTHING=20` は「仮想的に20件の全体平均サンプルを足す」という意味。件数が20より十分多いカテゴリではほぼ生の平均、少ないカテゴリでは全体平均に近づきます。

**使うとどうなるか（注意点）**
- 効果は大きいが、**foldの切り方に敏感**。Stratifiedにしないとfold間でカテゴリ平均がぶれます。
- カテゴリ数が少なくて件数が多い列（例: gender）ではほとんど効果がなく、木が自力で学べます。効くのは高カーディナリティ列。


In [ ]:
ENC_COLS = ['age', 'gender', 'stress_level', 'academic_work_impact', 
            'notifications_per_day', 'app_opens_per_day']

SMOOTHING = 20.0
global_mean = float(y.mean())

def get_levels(df):
    return pd.DataFrame({c: df[c].astype(object).fillna('__missing__').astype(str).values for c in ENC_COLS})

levels_tr = get_levels(train_base)
levels_te = get_levels(test_base)

def compute_target_encodings(train_idx, val_idx):
    tr_lvl = levels_tr.iloc[train_idx]
    va_lvl = levels_tr.iloc[val_idx]
    te_lvl = levels_te
    
    y_tr = y[train_idx]
    
    enc_tr = pd.DataFrame(index=train_idx)
    enc_va = pd.DataFrame(index=val_idx)
    enc_te = pd.DataFrame(index=range(len(test_base)))
    
    for c in ENC_COLS:
        stats = pd.DataFrame({'val': tr_lvl[c].values, 'y': y_tr}).groupby('val')['y'].agg(['count', 'mean'])
        smooth_map = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        
        enc_tr[f'te_{c}'] = tr_lvl[c].map(smooth_map).fillna(global_mean).astype(np.float32).values
        enc_va[f'te_{c}'] = va_lvl[c].map(smooth_map).fillna(global_mean).astype(np.float32).values
        enc_te[f'te_{c}'] = te_lvl[c].map(smooth_map).fillna(global_mean).astype(np.float32).values
        
    return enc_tr.reset_index(drop=True), enc_va.reset_index(drop=True), enc_te.reset_index(drop=True)

print('Target Encoding Engine ready for OOF execution.')

## 4. Stratified 5-Fold Multi-Model Training & Model Serialization

## 【解説】セル4: Stratified 5-Fold での3モデル学習とシリアライズ

**何をしているか（What）**
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` で5分割し、各foldで LightGBM（3500本）/ XGBoost / CatBoost を学習します。各foldごとに、

- 前セルのOOF Target Encodingを計算して特徴量に付加
- 検証foldに対する予測を `oof_*` 配列へ格納
- test に対する予測を5fold平均で `test_preds_*` へ蓄積
- 学習済みモデルを `saved_models/` に joblib / native形式で保存

**なぜそうするのか（Why）**
- **Stratified（層化）にする理由**: 目的変数の陽性率をfold間で揃えるため。不均衡データでランダム分割すると、foldによって陽性率がぶれてAUCの推定分散が大きくなります。
- **OOF予測を貯める理由**: 後段のブレンド重み最適化には「訓練データ全件に対する、リークのない予測」が必要です。OOFはそれを提供します。**OOFはアンサンブルの通貨**と言われる所以。
- **testを5fold平均する理由**: 単一モデルより分散が小さくなり、fold固有のノイズが打ち消し合います（bagging効果）。
- **モデル保存の理由**: 再学習に30分以上かかるため、後で重みだけ変えて試すときに再利用できます。

**用語メモ**
- **OOF (Out-Of-Fold) 予測**: 各サンプルについて「そのサンプルを学習に使わなかったモデル」が出した予測。訓練データ全体に対する擬似的な「未知データ性能」。
- **early stopping**: 検証スコアが一定ラウンド改善しなくなったら学習を打ち切る。木の本数を実質的に自動チューニングする役割。


In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cb  = np.zeros(len(train))

test_preds_lgb = np.zeros(len(test))
test_preds_xgb = np.zeros(len(test))
test_preds_cb  = np.zeros(len(test))

feature_cols = [c for c in train_base.columns if c not in ['id', TARGET]]
cat_cols = ['gender', 'stress_level', 'academic_work_impact']

print('=' * 65)
print('Executing 5-Fold Cross-Validation & Model Serialization...')
print('=' * 65)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_base, y)):
    t0 = time.time()
    print(f'\n--- [Fold {fold + 1}/{N_SPLITS}] ---')
    
    e_tr, e_va, e_te = compute_target_encodings(train_idx, val_idx)
    
    X_tr_base = train_base.iloc[train_idx][feature_cols].reset_index(drop=True)
    X_va_base = train_base.iloc[val_idx][feature_cols].reset_index(drop=True)
    X_te_base = test_base[feature_cols].reset_index(drop=True)
    
    X_tr = pd.concat([X_tr_base, e_tr], axis=1)
    X_va = pd.concat([X_va_base, e_va], axis=1)
    X_te = pd.concat([X_te_base, e_te], axis=1)
    
    y_tr, y_va = y[train_idx], y[val_idx]
    
    # ---------------- 1. LightGBM (Tuned) ----------------
    model_lgb = lgb.LGBMClassifier(
        n_estimators=3500,
        learning_rate=0.03,
        num_leaves=127,
        max_depth=8,
        colsample_bytree=0.75,
        subsample=0.8,
        subsample_freq=1,
        min_child_samples=50,
        reg_alpha=1.0,
        reg_lambda=5.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(60, verbose=False)]
    )
    pred_va_lgb = model_lgb.predict_proba(X_va)[:, 1]
    oof_lgb[val_idx] = pred_va_lgb
    test_preds_lgb += model_lgb.predict_proba(X_te)[:, 1] / N_SPLITS
    print(f'Fold {fold + 1} LightGBM ROC AUC: {roc_auc_score(y_va, pred_va_lgb):.5f}')
    joblib.dump(model_lgb, f'saved_models/lgb_fold_{fold+1}.joblib')
    
    # ---------------- 2. XGBoost (Tuned Hist) ----------------
    X_tr_xgb = X_tr.copy()
    X_va_xgb = X_va.copy()
    X_te_xgb = X_te.copy()
    for c in cat_cols:
        X_tr_xgb[c] = X_tr_xgb[c].cat.codes
        X_va_xgb[c] = X_va_xgb[c].cat.codes
        X_te_xgb[c] = X_te_xgb[c].cat.codes
        
    xgb_device = 'cuda' if HAS_GPU else 'cpu'
    model_xgb = xgb.XGBClassifier(
        n_estimators=3500,
        learning_rate=0.03,
        max_depth=7,
        colsample_bytree=0.75,
        subsample=0.8,
        reg_alpha=1.0,
        reg_lambda=4.0,
        tree_method='hist',
        device=xgb_device,
        random_state=SEED + fold,
        early_stopping_rounds=60,
        eval_metric='auc'
    )
    model_xgb.fit(
        X_tr_xgb, y_tr,
        eval_set=[(X_va_xgb, y_va)],
        verbose=False
    )
    pred_va_xgb = model_xgb.predict_proba(X_va_xgb)[:, 1]
    oof_xgb[val_idx] = pred_va_xgb
    test_preds_xgb += model_xgb.predict_proba(X_te_xgb)[:, 1] / N_SPLITS
    print(f'Fold {fold + 1} XGBoost  ROC AUC: {roc_auc_score(y_va, pred_va_xgb):.5f}')
    model_xgb.save_model(f'saved_models/xgb_fold_{fold+1}.json')
    
    # ---------------- 3. CatBoost (Native Categoricals) ----------------
    X_tr_cb = X_tr.copy()
    X_va_cb = X_va.copy()
    X_te_cb = X_te.copy()
    for c in cat_cols:
        X_tr_cb[c] = train_base.iloc[train_idx][c].astype(object).fillna('__missing__').astype(str).values
        X_va_cb[c] = train_base.iloc[val_idx][c].astype(object).fillna('__missing__').astype(str).values
        X_te_cb[c] = test_base[c].astype(object).fillna('__missing__').astype(str).values
        
    cb_task = 'GPU' if HAS_GPU else 'CPU'
    model_cb = cb.CatBoostClassifier(
        iterations=3500,
        learning_rate=0.035,
        depth=6,
        l2_leaf_reg=5.0,
        cat_features=cat_cols,
        task_type=cb_task,
        random_seed=SEED + fold,
        verbose=0
    )
    model_cb.fit(
        X_tr_cb, y_tr,
        eval_set=(X_va_cb, y_va),
        early_stopping_rounds=60,
        verbose=False
    )
    pred_va_cb = model_cb.predict_proba(X_va_cb)[:, 1]
    oof_cb[val_idx] = pred_va_cb
    test_preds_cb += model_cb.predict_proba(X_te_cb)[:, 1] / N_SPLITS
    print(f'Fold {fold + 1} CatBoost ROC AUC: {roc_auc_score(y_va, pred_va_cb):.5f}  (Elapsed: {time.time() - t0:.1f}s)')
    model_cb.save_model(f'saved_models/catboost_fold_{fold+1}.cbm')

print('=' * 65)
score_lgb = roc_auc_score(y, oof_lgb)
score_xgb = roc_auc_score(y, oof_xgb)
score_cb  = roc_auc_score(y, oof_cb)
print(f'Overall Out-of-Fold LightGBM ROC AUC: {score_lgb:.6f}')
print(f'Overall Out-of-Fold XGBoost  ROC AUC: {score_xgb:.6f}')
print(f'Overall Out-of-Fold CatBoost ROC AUC: {score_cb:.6f}')
print('=' * 65)

# Save OOF Predictions
oof_df = pd.DataFrame({
    'id': train['id'],
    'oof_lgb': oof_lgb,
    'oof_xgb': oof_xgb,
    'oof_cb': oof_cb
})
oof_df.to_csv('oof_predictions.csv', index=False)
print('OOF predictions successfully saved to oof_predictions.csv')

## 5. Master Calibrated Regime-Rank Fusion & Duplicate Magic (LB 0.97125+)

## 【解説】セル5: ランク融合・外部アンカー・Duplicate Magic（**要注意セル**）

**何をしているか（What）**
このセルは3段階に分かれます。

**(A) 自前3モデルのランク融合**
`to_logit(p)` で確率をロジット（`log(p/(1-p))`）に変換 → `percentile_rank` でパーセンタイル順位に変換 → SLSQP（逐次二次計画法）で `-AUC` を最小化する重み `(w_lgb, w_xgb, w_cb)` を求める（合計1、各0〜1の制約付き）。

**(B) 外部「SOTAアンカー」の自動検出と融合**
`/kaggle/input/**/*.csv` を全走査し、「行数がtestと一致し、`id` 列がtestと完全一致し、`addicted_label` 列を持つ」CSVを探します。これは実質的に**アタッチされた他人の公開submissionファイル**です。見つかったら:

```
final = 0.725 × アンカー1 + 0.225 × アンカー2 + 0.050 × 自前ブレンド
```

**(C) Duplicate Magic**
ID `735378` を 1.0、ID `862871` を 0.0 に決め打ちで上書き。

**なぜそうするのか（Why）— そして何を学ぶべきか**

- **(A) のランク融合は正当かつ有用な技術**です。AUCは順位のみを見るので、スケールの異なるモデル出力を順位に揃えてから混ぜるのは理論的に正しい。指標そのものを目的関数に据えて重みを最適化するのも、代理損失を経由しない直接的なアプローチとして学ぶ価値があります。

- **(B) が本notebookのスコアの正体**です。自前で作った特徴量・3モデル・5-fold CVの寄与は、最終予測のわずか **5%** です。残り95%は他人が公開した予測の重み付き平均。これは Playground Series で「public blend」と呼ばれる広く行われている行為で、ルール違反ではありませんが、**次の意味でスコアが実力を表さなくなります**:
  - LB 0.97125 という数字から「この特徴量エンジニアリングが優れている」とは**まったく言えない**。アンカーを外した純粋な自前スコアは、このnotebookからは分かりません。
  - 全員が同じ公開アンカーを混ぜるため、**上位が同一の予測に収束**し、private LBでは全員がまとめて順位を落とす（あるいは順位がシャッフルされる）現象が起きます。
  - `if sota_anchors:` の分岐により、**アンカーCSVをアタッチせずに実行すると結果が変わる**。再現性の観点でも脆い設計です。

- **(C) Duplicate Magic は最も危険**です。2件のIDをラベル決め打ちしています。これは「trainとtestに完全一致する重複行があり、そのラベルを転記できる」という**リーク利用**の典型です。AUCへの寄与は2/testサイズなのでほぼ無視できる程度ですが、思想として「データ生成の穴を突く」方向であり、汎化性能とは無関係です。

**この解説の目的**: 高スコアnotebookを読むときは、**「そのスコアはどこから来たのか」を分解して確認する**のが最重要のスキルです。数字だけを見て手法を信用しないこと。本notebookは (A) の技術を学ぶ教材としては優秀で、(B)(C) は「Kaggleでこういうことが起きている」という現象の観察対象として扱うのが適切です。

**用語メモ**
- **SLSQP (Sequential Least SQuares Programming)**: 等式・不等式制約付き非線形最適化を解く手法。「重みの合計＝1」のような制約を扱える。
- **パーセンタイルランク**: 値を順位に直し、`(順位 - 0.5) / N` で0〜1に正規化したもの。分布形状に依らず一様分布になる。
- **public blend**: 公開されている他人の予測を混ぜてスコアを上げる行為。Kaggleで合法だが、private shakeup（順位大変動）の主因でもある。


In [ ]:
def percentile_rank(values):
    values = np.asarray(values, dtype=np.float64)
    return (rankdata(values, method='average') - 0.5) / len(values)

def to_logit(p, clip=30.0):
    p = np.clip(np.asarray(p, np.float64), 1e-12, 1.0 - 1e-12)
    return np.clip(np.log(p / (1.0 - p)), -clip, clip)

# Local Model Zoo Percentile Ranks
rank_oof_lgb = percentile_rank(to_logit(oof_lgb))
rank_oof_xgb = percentile_rank(to_logit(oof_xgb))
rank_oof_cb  = percentile_rank(to_logit(oof_cb))

rank_te_lgb = percentile_rank(to_logit(test_preds_lgb))
rank_te_xgb = percentile_rank(to_logit(test_preds_xgb))
rank_te_cb  = percentile_rank(to_logit(test_preds_cb))

# Optimize Local Zoo Weights
def objective(weights):
    w1, w2, w3 = weights
    blend = w1 * rank_oof_lgb + w2 * rank_oof_xgb + w3 * rank_oof_cb
    return -roc_auc_score(y, blend)

init_w = [0.45, 0.35, 0.20]
bounds = [(0, 1), (0, 1), (0, 1)]
constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})

opt_res = minimize(objective, init_w, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = opt_res.x / np.sum(opt_res.x)
local_oof_auc = -opt_res.fun

print(f'Optimal Local Weights -> LightGBM: {best_weights[0]:.4f} | XGBoost: {best_weights[1]:.4f} | CatBoost: {best_weights[2]:.4f}')
print(f'Local Out-of-Fold ROC AUC: {local_oof_auc:.6f}')
joblib.dump(best_weights, 'saved_models/ensemble_weights.joblib')

local_test_blend = (best_weights[0] * rank_te_lgb + 
                    best_weights[1] * rank_te_xgb + 
                    best_weights[2] * rank_te_cb)
local_test_rank = percentile_rank(local_test_blend)

# ---------------- Regime Anchor Detection & Calibration ----------------
regime_candidates = glob.glob('/kaggle/input/**/*.csv', recursive=True)
regime_candidates = [f for f in regime_candidates if not any(x in f for x in ['train.csv', 'test.csv', 'sample_submission.csv', 'importance'])]

sota_anchors = []
for f in regime_candidates:
    try:
        df_cand = pd.read_csv(f)
        if len(df_cand) == len(test) and 'addicted_label' in df_cand.columns and 'id' in df_cand.columns:
            if df_cand['id'].equals(test['id']) and np.isfinite(df_cand['addicted_label']).all():
                print(f'Discovered High-Performance SOTA Regime Anchor: {f}')
                sota_anchors.append(percentile_rank(df_cand['addicted_label'].values))
    except Exception as e:
        pass

if sota_anchors:
    print(f'Applying Grandmaster Regime Fusion across {len(sota_anchors)} anchor(s)...')
    primary_anchor = sota_anchors[0]
    secondary_anchor = sota_anchors[1] if len(sota_anchors) > 1 else primary_anchor
    
    # Grandmaster Calibration Formula (72.5% Primary + 22.5% Secondary + 5.0% Local Diverse Zoo)
    final_raw = 0.725 * primary_anchor + 0.225 * secondary_anchor + 0.050 * local_test_rank
    final_predictions = percentile_rank(final_raw)
else:
    print('No external anchors found. Outputting purely trained local model zoo.')
    final_predictions = local_test_rank

# ---------------- Deterministic Duplicate Magic ----------------
submission = pd.DataFrame({
    'id': test['id'],
    'addicted_label': final_predictions
})

def apply_duplicate_magic(df):
    if 735378 in df['id'].values:
        df.loc[df['id'] == 735378, 'addicted_label'] = 1.0
    if 862871 in df['id'].values:
        df.loc[df['id'] == 862871, 'addicted_label'] = 0.0
    return df

submission = apply_duplicate_magic(submission)
submission.to_csv('submission.csv', index=False)
print('Master submission generated and saved to submission.csv (LB 0.97125+ Ready!)')
print(f'Submission shape: {submission.shape}')
display(submission.head(10))

## 6. Global Feature Importance Interpretation

## 【解説】セル6: 特徴量重要度の可視化

**何をしているか（What）**
最後のfoldのLightGBMモデルから `feature_importances_`（デフォルトは split gain = その特徴で分割した回数）を取り出し、上位25件を横棒グラフで表示、全件をCSVに保存します。

**なぜそうするのか（Why）**
- どの特徴が効いたかを見ることで、FEの仮説が正しかったかを検証できます。ここで `frac_*` や `d1_*`（小数格子）が上位に来ていれば、**このデータが合成であることの証拠**が可視化されたことになります。
- 逆に効いていない特徴は削ることで、学習時間短縮とわずかな過学習抑制が期待できます。

**注意点**
- `split` ベースの重要度は「連続値で分割候補が多い特徴」を過大評価する既知のバイアスがあります。より公平に見たいなら `importance_type='gain'` や **SHAP値**を使うべきです。
- 重要度は**因果ではなく相関**です。「小数第1位が重要」＝「小数第1位が依存を引き起こす」ではありません。


In [ ]:
feat_imp = pd.DataFrame({
    'feature': model_lgb.feature_name_,
    'importance': model_lgb.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(x='importance', y='feature', data=feat_imp.head(25), palette='mako')
plt.title('Top 25 Most Important Features across Master Pipeline', fontsize=14, fontweight='bold')
plt.xlabel('Importance (Split Gain)')
plt.tight_layout()
plt.show()

feat_imp.to_csv('feature_importance.csv', index=False)
print('Saved feature importances to feature_importance.csv')